In [ ]:
import torch
import time
import wandb
import numpy as np
import torch.nn as nn
import gymnasium as gym
import ale_py
from gymnasium.vector import SyncVectorEnv
from gymnasium.wrappers import RecordVideo

# **configurations**

In [ ]:
GAME_ID = "Taxi-v4"
TOTAL_TRAJECTORIES = 10_000
K_PARRALLEL_ROLLOUTS = 12
MAX_ROLLOUT = 200
EPOCHS = 5
BATCH_SIZE = 128
EVAL_STEPS = 50
EVAL_LOOP = 3
FIXED_SEED = 54
LEARNING_RATE = 3e-4

# ========== ALGORITHM PARAMETER =============#
ENTROPY_BETA = 0.01
CLIPPED_VALUE = 0.2

WANDB_API_KEY = " "
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def wandb_runs():
  wandb.login(key=WANDB_API_KEY)
  run = wandb.init(
    project="GRPO",
    name = "GRPO-Taxi"
    )
  return run

In [ ]:
def make_environment():
  def _init():
    env = gym.make(GAME_ID)
    return env
  return _init

In [ ]:
env = SyncVectorEnv([make_environment() for _ in range(K_PARRALLEL_ROLLOUTS)], autoreset_mode=gym.vector.AutoresetMode.DISABLED)

In [ ]:
#env.single_observation_space
env.reset()[0]

In [ ]:
class PolicyNetwork(nn.Module):

    def __init__(self):
        super().__init__()

        self.StateEmbedding = nn.Embedding(500, 128)
        self.linear_layer = nn.Sequential(
            nn.Linear(in_features=128, out_features=128),
            nn.ReLU(),
            nn.Linear(in_features=128, out_features=env.single_action_space.n)
        )

    def forward(self, x):
        x = self.StateEmbedding(x)
        x = self.linear_layer(x)
        return x

    def choose_action(self, x, chosen_actions = None):
        logits = self(x)
        dist = torch.distributions.Categorical(logits=logits)

        if chosen_actions is None:
          actions = dist.sample()

        else:
          actions = chosen_actions

        log_probs = dist.log_prob(actions)
        entropy = dist.entropy()

        return actions, log_probs, entropy

In [ ]:
actornetwork = PolicyNetwork().to(DEVICE)

In [ ]:
sum(p.numel() for p in actornetwork.parameters())

# **EVALUATION**

In [ ]:
def evaluation(actornet, rec_video = False):

  eval_env = gym.make(GAME_ID, render_mode = 'rgb_array')

  if rec_video:
    video_dir = f"videos/{int(time.time())}"
    eval_env = RecordVideo(eval_env, video_folder=video_dir, episode_trigger=lambda ep: True)

  total_reward = 0
  total_step = 0

  with torch.no_grad():

    for _ in range(EVAL_LOOP):

      state = eval_env.reset()[0]
      done = False
      ep_reward = 0

      while not done:
        state = torch.tensor(state, dtype=torch.long, device=DEVICE).unsqueeze(0)
        action = actornet(state).argmax().item()
        next_state, reward, terminated, truncation, _ = eval_env.step(action)
        done = terminated or truncation
        state = next_state
        total_reward += float(reward)
        total_step += 1
        ep_reward += float(reward)

      print(ep_reward)


  average_reward = total_reward / EVAL_LOOP
  average_step = total_step / EVAL_LOOP
  eval_env.close()

  return average_reward, average_step

In [ ]:
evaluation(actornetwork,True)

In [ ]:
actor_optimizer = torch.optim.Adam(actornetwork.parameters(), lr=LEARNING_RATE)

In [ ]:
runs = wandb_runs()

# **GRPO TRAINING LOOP**

In [ ]:
for step in range(TOTAL_TRAJECTORIES):

    rollout = 0

    state, _  = env.reset(seed = [FIXED_SEED + 2 * step] * K_PARRALLEL_ROLLOUTS)

    training_reward = torch.zeros(K_PARRALLEL_ROLLOUTS).to(DEVICE)

    buffer_states = []
    buffer_actions = []
    buffer_rewards = []
    buffer_dones = []
    buffer_log_probs = []

    status = False

    while rollout <= MAX_ROLLOUT or (not status):

      state_T = torch.tensor(state, dtype=torch.long).to(DEVICE)

      with torch.no_grad():
        actions, log_probs, entropy = actornetwork.choose_action(state_T)

      next_state, reward, terminated, truncated, _ = env.step(actions.cpu().numpy())

      done = terminated | truncated

      done_T = torch.tensor(done, dtype=torch.float).to(DEVICE).detach()

      status = done_T.any().item()

      training_reward += torch.tensor(reward, dtype=torch.float).to(DEVICE)

      buffer_states.append(state_T.detach())
      buffer_actions.append(actions.detach())
      buffer_rewards.append(torch.tensor(reward, dtype=torch.float).to(DEVICE).detach())
      buffer_dones.append(done_T)
      buffer_log_probs.append(log_probs.detach())

      if done.any():
        reset_mask = done.astype(bool)
        next_state, _ = env.reset(options={"reset_mask": reset_mask})   # only resets the one with "TRUE" and wont touches others.

      state = next_state
      rollout += 1

    # ============================ GRPO ============================ #

    all_states = torch.stack(buffer_states).permute(1, 0)
    all_actions = torch.stack(buffer_actions).permute(1, 0)
    all_rewards = torch.stack(buffer_rewards).permute(1, 0)
    all_dones = torch.stack(buffer_dones).permute(1, 0)
    old_log_probs = torch.stack(buffer_log_probs).permute(1, 0)

    already_done = torch.zeros_like(all_dones)
    already_done[:, 1:] = all_dones[:, :-1]      # shift right by 1
    mask = (1 - already_done.cumsum(dim=-1)).clamp(min=0).int()

    mask_idx = mask.sum(dim = -1)                                   # [8, 9, 11, 13]  for each trajectory

    traj_states = torch.cat([all_states[i, :mask_idx[i], ...] for i in range(all_states.size(0))])

    traj_actions = torch.cat([all_actions[i, :mask_idx[i]] for i in range(all_states.size(0))])

    traj_log_probs = torch.cat([old_log_probs[i, :mask_idx[i]] for i in range(all_states.size(0))])

    actual_rewards = mask * all_rewards

    sum_returns = actual_rewards.sum(dim = -1)          # sum over the each trajectoris
    mean_returns = sum_returns.mean()
    std_returns = sum_returns.std()

    individual_advantages = ((sum_returns - mean_returns) / (std_returns + 1e-7))

    advantages = torch.cat([individual_advantages[i].unsqueeze(0).repeat(mask_idx[i]) for i in range(individual_advantages.size(0))])

    running_policy_loss = 0
    running_entropy_loss = 0
    running_step = 0

    for epoch in range(EPOCHS):

        RandomIndexes = torch.randperm(traj_states.size(0))


        for i in range(0, RandomIndexes.size(0), BATCH_SIZE):

          randIdx = RandomIndexes[i : i + BATCH_SIZE]

          mb_states = traj_states[randIdx]
          mb_actions = traj_actions[randIdx]
          mb_advantages = advantages[randIdx]
          mb_old_log_probs = traj_log_probs[randIdx]

          _, new_log_probs, new_entropy = actornetwork.choose_action(mb_states, mb_actions)

          ratio = torch.exp(new_log_probs - mb_old_log_probs)

          policy_inverse_loss = torch.min(ratio * mb_advantages, torch.clamp(ratio, 1 - CLIPPED_VALUE, 1 + CLIPPED_VALUE) * mb_advantages).mean()

          entropy_loss = ENTROPY_BETA * new_entropy.mean()

          loss_func = - (policy_inverse_loss + entropy_loss)

          actor_optimizer.zero_grad()

          loss_func.backward()

          actor_optimizer.step()

          running_policy_loss += policy_inverse_loss.item()
          running_entropy_loss += entropy_loss.item()
          running_step += 1

    runs.log({
        "step": step,
        "policy_inverse_loss": running_policy_loss/running_step,
        "entropy_loss": running_entropy_loss/running_step,
        "advantages-mean": advantages.mean().item(),
        "advantages-std": advantages.std().item(),
        "average_training_reward": training_reward.mean().item(),
        "max_training_reward": training_reward.max().item(),
        "min_training_reward": training_reward.min().item()
    })

    if step%EVAL_STEPS == 0 and step>0:
          actornetwork.eval()
          average_reward, average_step = evaluation(actornet=actornetwork)
          actornetwork.train()

          runs.log({
              "average_reward": average_reward,
              "average_step": average_step,
          })